In [1]:
# Updated to use Random Forest Classifier on mean and std of wav files
# Alex Stedman
# We will extract mean and std from wav files for classification
import os
import numpy as np
import pandas as pd
import wave
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Specify training and validation directories
train_dir = "../Datasets/SmellySongs9k_V2/train"
val_dir = "../Datasets/SmellySongs9k_V2/test"

# Function to extract mean and std from wav files
def extract_features(wav_path):
    with wave.open(wav_path, 'rb') as wav_file:
        n_frames = wav_file.getnframes()
        frames = wav_file.readframes(n_frames)
        samples = np.frombuffer(frames, dtype=np.int16)
        return np.mean(samples), np.std(samples)

# Collect features and labels
def collect_data(directory):
    data = []
    labels = []
    for label in os.listdir(directory):
        label_dir = os.path.join(directory, label)
        if os.path.isdir(label_dir):
            for file in os.listdir(label_dir):
                if file.endswith('.wav'):
                    wav_path = os.path.join(label_dir, file)
                    mean, std = extract_features(wav_path)
                    data.append([mean, std])
                    labels.append(label)
    return np.array(data), np.array(labels)

# Collect training and validation data
X_train, y_train = collect_data(train_dir)
X_val, y_val = collect_data(val_dir)

# Convert labels to integers
label_mapping = {label: idx for idx, label in enumerate(np.unique(y_train))}
y_train = np.array([label_mapping[label] for label in y_train])
y_val = np.array([label_mapping[label] for label in y_val])

# Train Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate the model
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred, target_names=label_mapping.keys()))

              precision    recall  f1-score   support

          AI       0.89      0.82      0.85       981
       Human       0.84      0.89      0.86       981

    accuracy                           0.86      1962
   macro avg       0.86      0.86      0.86      1962
weighted avg       0.86      0.86      0.86      1962

